# Areal Reduction Factors (ARF) — Computation and Application

Point precipitation from NOAA Atlas 14 (or similar DDF tables) applies at a
single gauge location.  For larger drainage areas the *spatially averaged*
precipitation is always less than the point value.  An **Areal Reduction
Factor (ARF)** accounts for this: multiply the point depth by the ARF to
obtain the mean-areal depth over the watershed.

This notebook walks through the full workflow:

| Step | API | Description |
|------|-----|-------------|
| 1 | `lookup_arf_from_dar()` | Interpolate ARF from a DAR curve at a given CDA |
| 2 | `compute_kcda_cdas()` | Traverse upstream network, sum subbasin areas → CDA per junction |
| 3 | `build_kcda_arf_table()` | Combined: CDA + DAR lookup for a list of outlet junctions |
| 4 | `apply_arf()` | Multiply all `Depth:` values in a met file by an ARF scalar |

**Background — DAR Curve**:  A Depth-Area Reduction (DAR) curve plots ARF vs
contributing drainage area.  At CDA = 0 the ARF = 1.0 (no reduction); it
decreases smoothly with increasing area.  DAR curves are derived from
multi-storm analyses using tools like HEC-MetVue.

**Estimated Time**: 5 minutes

In [1]:
# pip install hms-commander

**For Development**: If working on hms-commander source code, use the `hmscmdr_local`
conda environment (editable install) instead of pip install.

In [2]:
import shutil
from pathlib import Path
import pandas as pd

from hms_commander import HmsExamples, HmsBasin, HmsArf

print("hms-commander loaded")

hms-commander loaded


## 1. Define a DAR Curve

A DAR curve maps contributing drainage area → ARF.  In practice this comes
from a historical storm analysis.  Here we define a representative 24-hour
curve as a list of `(area, arf)` tuples.

`lookup_arf_from_dar()` supports three input formats:
- **List of tuples**: `[(area, arf), ...]`
- **DataFrame**: columns `['area', 'arf']`
- **Duration-keyed dict**: `{duration_hours: [(area, arf), ...]}`

In [3]:
# 24-hour DAR curve (area in model units, e.g. sq mi or acres)
dar_24hr = [
    (0.5,    1.000),
    (1.0,    0.990),
    (2.0,    0.975),
    (5.0,    0.955),
    (10.0,   0.930),
    (25.0,   0.900),
    (50.0,   0.870),
    (100.0,  0.840),
]

# Convert to DataFrame for display
dar_df = pd.DataFrame(dar_24hr, columns=['area', 'arf'])
print("24-hour DAR curve:")
dar_df

24-hour DAR curve:


,area,arf
0,0.5,1.000
1,1.0,0.990
2,2.0,0.975
3,5.0,0.955
4,10.0,0.930
5,25.0,0.900
6,50.0,0.870
7,100.0,0.840


In [4]:
# Demonstrate interpolation at specific CDA values
test_cdas = [0.3, 1.5, 3.5, 7.5, 20.0, 75.0]

print(f"{'CDA':>10}  {'ARF':>8}")
print("-" * 22)
for cda in test_cdas:
    arf = HmsArf.lookup_arf_from_dar(cda, dar_24hr)
    print(f"{cda:>10.1f}  {arf:>8.4f}")

print()
print("Note: CDA <= 0.5 (minimum curve area) returns 1.0 (no reduction)")

       CDA       ARF
----------------------
       0.3    1.0000
       1.5    0.9825
       3.5    0.9650
       7.5    0.9425
      20.0    0.9100
      75.0    0.8550

Note: CDA <= 0.5 (minimum curve area) returns 1.0 (no reduction)


## 2. Extract Example Basin

We use the **Castro Valley** example project (4 subbasins, 2 branch junctions,
1 outlet).  Any HMS basin file works — the traversal logic is topology-based.

In [5]:
project_path = HmsExamples.extract_project(
    "castro",
    output_path=Path.cwd() / 'hms_example_projects' / 'castro_arf'
)

basin_file = next(Path(project_path).glob('*.basin'))
print(f"Basin file: {basin_file.name}")

# Review the basin network
subbasins = HmsBasin.get_subbasins(basin_file)
junctions = HmsBasin.get_junctions(basin_file)
reaches   = HmsBasin.get_reaches(basin_file)

print(f"\n{len(subbasins)} subbasins, {len(junctions)} junctions, {len(reaches)} reaches")
print("\nSubbasins:")
print(subbasins[['name', 'area', 'downstream']].to_string(index=False))
print("\nJunctions (analysis points):")
print(junctions[['name', 'downstream']].to_string(index=False))

2026-03-07 15:08:04 - hms_commander.HmsExamples - INFO - Found HMS 4.10 at C:\Program Files\HEC\HEC-HMS\4.10


2026-03-07 15:08:04 - hms_commander.HmsExamples - INFO - Found HMS 4.11 at C:\Program Files\HEC\HEC-HMS\4.11


2026-03-07 15:08:04 - hms_commander.HmsExamples - INFO - Found HMS 4.12 at C:\Program Files\HEC\HEC-HMS\4.12


2026-03-07 15:08:04 - hms_commander.HmsExamples - INFO - Found HMS 4.13 at C:\Program Files\HEC\HEC-HMS\4.13


2026-03-07 15:08:04 - hms_commander.HmsExamples - INFO - Found HMS 4.4.1 at C:\Program Files\HEC\HEC-HMS\4.4.1


...

2026-03-07 15:08:04 - hms_commander.HmsExamples - INFO - Destination: C:\GH\hms-commander\examples\hms_example_projects\castro_arf\castro


2026-03-07 15:08:04 - hms_commander.HmsExamples - INFO - Successfully extracted 'castro' to C:\GH\hms-commander\examples\hms_example_projects\castro_arf\castro


2026-03-07 15:08:04 - hms_commander.HmsBasin - INFO - Reading subbasins from: C:\GH\hms-commander\examples\hms_example_projects\castro_arf\castro\Castro_1.basin


2026-03-07 15:08:04 - hms_commander.HmsBasin - INFO - Found 4 subbasins


Basin file: Castro_1.basin

4 subbasins, 3 junctions, 2 reaches

Subbasins:
      name  area  downstream
Subbasin-3  2.17     Reach-2
Subbasin-4  0.96 West Branch
Subbasin-1  0.86     Reach-1
Subbasin-2  1.52 East Branch

Junctions (analysis points):
       name downstream
West Branch     Outlet
East Branch     Outlet
     Outlet       None


## 3. Compute CDAs at Outlet Junctions

`compute_kcda_cdas()` traverses the upstream network from each named junction
and sums the contributing subbasin areas.  The area unit matches the HMS model
(English = square miles; Metric = square kilometres).

In [6]:
# Use all three junctions as analysis points
outlet_junctions = ["West Branch", "East Branch", "Outlet"]

cda_df = HmsArf.compute_kcda_cdas(basin_file, outlet_junctions)
print("Contributing Drainage Areas:")
print(cda_df[['junction', 'cda_acres', 'subbasin_count']].to_string(index=False))

2026-03-07 15:08:04 - hms_commander.HmsBasin - INFO - Reading subbasins from: C:\GH\hms-commander\examples\hms_example_projects\castro_arf\castro\Castro_1.basin


2026-03-07 15:08:04 - hms_commander.HmsBasin - INFO - Found 4 subbasins


2026-03-07 15:08:04 - hms_commander.HmsBasin - INFO - Building upstream network from: C:\GH\hms-commander\examples\hms_example_projects\castro_arf\castro\Castro_1.basin


2026-03-07 15:08:04 - hms_commander.HmsBasin - INFO - Built upstream network: 5 targets, 8 upstream connections


2026-03-07 15:08:04 - hms_commander.HmsBasin - INFO - Upstream of 'West Branch': 2 subbasins, 0 junctions, 0 reaches, 0 diversions


...

2026-03-07 15:08:04 - hms_commander.HmsBasin - INFO - Upstream of 'East Branch': 2 subbasins, 0 junctions, 0 reaches, 0 diversions


2026-03-07 15:08:04 - hms_commander.HmsBasin - INFO - Building upstream network from: C:\GH\hms-commander\examples\hms_example_projects\castro_arf\castro\Castro_1.basin


2026-03-07 15:08:04 - hms_commander.HmsBasin - INFO - Built upstream network: 5 targets, 8 upstream connections


2026-03-07 15:08:04 - hms_commander.HmsBasin - INFO - Upstream of 'Outlet': 4 subbasins, 2 junctions, 0 reaches, 0 diversions


Contributing Drainage Areas:
   junction  cda_acres  subbasin_count
West Branch       3.13               2
East Branch       2.38               2
     Outlet       5.51               4


In [7]:
# The upstream_subbasins column shows which subbasins were counted
print("Upstream subbasins per junction:")
for _, row in cda_df.iterrows():
    print(f"  {row['junction']}: {sorted(row['upstream_subbasins'])}")

Upstream subbasins per junction:
  West Branch: ['Subbasin-3', 'Subbasin-4']
  East Branch: ['Subbasin-1', 'Subbasin-2']
  Outlet: ['Subbasin-1', 'Subbasin-2', 'Subbasin-3', 'Subbasin-4']


## 4. Build the Full ARF Table

`build_kcda_arf_table()` combines the CDA computation and DAR lookup into a
single call, returning results sorted by ascending CDA.  The `arf` column is
the multiplier to apply to point precipitation depths.

In [8]:
arf_table = HmsArf.build_kcda_arf_table(
    basin_file,
    outlet_junctions,
    dar_curve=dar_24hr,
    duration_hours=24,
)

print("ARF table (sorted by CDA ascending):")
arf_table[['junction', 'cda_acres', 'subbasin_count', 'arf']]

2026-03-07 15:08:04 - hms_commander.HmsBasin - INFO - Reading subbasins from: C:\GH\hms-commander\examples\hms_example_projects\castro_arf\castro\Castro_1.basin


2026-03-07 15:08:04 - hms_commander.HmsBasin - INFO - Found 4 subbasins


2026-03-07 15:08:04 - hms_commander.HmsBasin - INFO - Building upstream network from: C:\GH\hms-commander\examples\hms_example_projects\castro_arf\castro\Castro_1.basin


2026-03-07 15:08:04 - hms_commander.HmsBasin - INFO - Built upstream network: 5 targets, 8 upstream connections


2026-03-07 15:08:04 - hms_commander.HmsBasin - INFO - Upstream of 'West Branch': 2 subbasins, 0 junctions, 0 reaches, 0 diversions


...

2026-03-07 15:08:04 - hms_commander.HmsBasin - INFO - Building upstream network from: C:\GH\hms-commander\examples\hms_example_projects\castro_arf\castro\Castro_1.basin


2026-03-07 15:08:04 - hms_commander.HmsBasin - INFO - Built upstream network: 5 targets, 8 upstream connections


2026-03-07 15:08:04 - hms_commander.HmsBasin - INFO - Upstream of 'Outlet': 4 subbasins, 2 junctions, 0 reaches, 0 diversions


ARF table (sorted by CDA ascending):


,junction,cda_acres,subbasin_count,arf
0,East Branch,2.38,2,0.972467
1,West Branch,3.13,2,0.967467
2,Outlet,5.51,4,0.952450


In [9]:
# 'a14_dar_multiplier' is an alias for 'arf' — same value, used in report headers
print("arf == a14_dar_multiplier:",
      (arf_table['arf'] == arf_table['a14_dar_multiplier']).all())

arf == a14_dar_multiplier: True


## 5. Apply ARF to a Met File

`apply_arf(arf=scalar)` multiplies every `Depth:` line inside the
`Precip Method Parameters:` block by the given scalar.  This is the correct
target for **Frequency Based Hypothetical** met files where all subbasin
blocks are empty and depths are stored globally.

The method creates a `.met.bak` backup before any modification.

In [10]:
# Create a minimal Frequency Based Hypothetical met file to demonstrate apply_arf
example_met_content = """Meteorology: 1PCT_24HR
     Last Modified Date: 01 January 2024
     Last Modified Time: 00:00:00
     Version: 4.11
     Unit System: English
     Precipitation Method: Frequency Based Hypothetical
     Snowmelt Method: None
End:

Precip Method Parameters: Frequency Based Hypothetical
     Exceedence Frequency: 1
     Single Hypothetical Storm Size: Yes
     Total Duration: 1440
     Time Interval: 5
     Percent of Duration Before Peak Rainfall: 67
     Depth: 1.2000
     Depth: 2.1000
     Depth: 4.3000
     Depth: 5.7000
     Depth: 6.8000
     Depth: 9.1000
     Depth: 11.100
     Depth: 13.500
End:

Subbasin: Subbasin-1
End:

Subbasin: Subbasin-2
End:

Subbasin: Subbasin-3
End:

Subbasin: Subbasin-4
End:
"""

# Write to a temporary location for this demo
work_dir = Path.cwd() / 'hms_example_projects' / 'arf_demo'
work_dir.mkdir(parents=True, exist_ok=True)
met_path = work_dir / '1PCT_24HR.met'
met_path.write_text(example_met_content, encoding='utf-8')

print(f"Created example met file: {met_path.name}")
print("\nOriginal Depth: values in Precip Method Parameters block:")

import re
depth_re = re.compile(r'^\s*Depth:\s*([\d.]+)', re.MULTILINE)
original_depths = [float(m.group(1)) for m in depth_re.finditer(example_met_content)]
print(original_depths)

Created example met file: 1PCT_24HR.met

Original Depth: values in Precip Method Parameters block:
[1.2, 2.1, 4.3, 5.7, 6.8, 9.1, 11.1, 13.5]


In [11]:
# Select ARF for the 'Outlet' junction (largest CDA → smallest ARF)
outlet_row = arf_table[arf_table['junction'] == 'Outlet'].iloc[0]
arf_scalar = outlet_row['arf']

print(f"Outlet junction CDA: {outlet_row['cda_acres']:.2f} sq mi")
print(f"ARF from DAR curve:  {arf_scalar:.4f}")

# Apply the ARF scalar to all depths in the met file
result = HmsArf.apply_arf(met_path, arf=arf_scalar)

print(f"\nDepths modified: {result['depths_modified']}")
print(f"Backup created:  {Path(result['backup_path']).name}")

2026-03-07 15:08:04 - hms_commander.HmsArf - INFO - Created backup: C:\GH\hms-commander\examples\hms_example_projects\arf_demo\1PCT_24HR.met.bak


2026-03-07 15:08:04 - hms_commander.HmsArf - INFO - ARF 0.9525 applied: 8 Depth: lines updated in 1PCT_24HR.met


Outlet junction CDA: 5.51 sq mi
ARF from DAR curve:  0.9525

Depths modified: 8
Backup created:  1PCT_24HR.met.bak


In [12]:
# Read back and compare before / after
modified_content = met_path.read_text(encoding='utf-8')
modified_depths = [float(m.group(1)) for m in depth_re.finditer(modified_content)]

comparison = pd.DataFrame({
    'original': original_depths,
    'modified': modified_depths,
    'ratio':    [m / o if o > 0 else float('nan')
                 for o, m in zip(original_depths, modified_depths)],
})

print(f"ARF applied: {arf_scalar:.4f}")
print()
comparison

ARF applied: 0.9525



,original,modified,ratio
0,1.2,1.1429,0.952417
1,2.1,2.0001,0.952429
2,4.3,4.0955,0.952442
3,5.7,5.4290,0.952456
4,6.8,6.4767,0.952456
5,9.1,8.6673,0.952451
6,11.1,10.5722,0.952450
7,13.5,12.8581,0.952452


## 6. Batch Workflow — Clone Then Apply

The recommended production pattern is **non-destructive**: clone the base met
model for each analysis junction and apply that junction's ARF scalar to the
clone.  This preserves the original and produces one ARF-adjusted met model
per junction for comparison in the HMS GUI.

In [13]:
from hms_commander import HmsMet

# Start from the original (unmodified backup → restore original)
base_met = work_dir / 'base_1PCT_24HR.met'
base_met.write_text(example_met_content, encoding='utf-8')

cloned_mets = {}

for _, row in arf_table.iterrows():
    junction = row['junction']
    arf_val  = row['arf']
    slug     = junction.replace(' ', '_')

    # Copy base met to a junction-specific file
    target = work_dir / f'1PCT_24HR_{slug}_ARF.met'
    shutil.copy2(base_met, target)

    # Apply ARF (no backup needed — file was just copied)
    r = HmsArf.apply_arf(target, arf=arf_val, preserve_original=False)
    cloned_mets[junction] = (target, arf_val, r['depths_modified'])

    print(f"{junction:15s}  ARF={arf_val:.4f}  → {target.name}")

print(f"\n{len(cloned_mets)} met models created")

2026-03-07 15:08:04 - hms_commander.HmsArf - INFO - ARF 0.9725 applied: 8 Depth: lines updated in 1PCT_24HR_East_Branch_ARF.met


2026-03-07 15:08:04 - hms_commander.HmsArf - INFO - ARF 0.9675 applied: 8 Depth: lines updated in 1PCT_24HR_West_Branch_ARF.met


2026-03-07 15:08:04 - hms_commander.HmsArf - INFO - ARF 0.9525 applied: 8 Depth: lines updated in 1PCT_24HR_Outlet_ARF.met


East Branch      ARF=0.9725  → 1PCT_24HR_East_Branch_ARF.met
West Branch      ARF=0.9675  → 1PCT_24HR_West_Branch_ARF.met
Outlet           ARF=0.9525  → 1PCT_24HR_Outlet_ARF.met

3 met models created


In [14]:
# Verify: show the 8-duration depths from each cloned met file
rows = []
for junction, (path, arf_val, _) in sorted(cloned_mets.items()):
    depths = [float(m.group(1))
              for m in depth_re.finditer(path.read_text(encoding='utf-8'))]
    row = {'junction': junction, 'arf': arf_val}
    for i, d in enumerate(depths, 1):
        row[f'd{i}'] = round(d, 4)
    rows.append(row)

summary_df = pd.DataFrame(rows)
print("Adjusted depths by junction (first 8 non-zero durations):")
summary_df

Adjusted depths by junction (first 8 non-zero durations):


,junction,arf,d1,d2,d3,d4,d5,d6,d7,d8
0,East Branch,0.972467,1.1670,2.0422,4.1816,5.5431,6.6128,8.8494,10.7944,13.1283
1,Outlet,0.952450,1.1429,2.0001,4.0955,5.4290,6.4767,8.6673,10.5722,12.8581
2,West Branch,0.967467,1.1610,2.0317,4.1601,5.5146,6.5788,8.8039,10.7389,13.0608


## 7. Cleanup

In [15]:
shutil.rmtree(Path.cwd() / 'hms_example_projects' / 'castro_arf', ignore_errors=True)
shutil.rmtree(Path.cwd() / 'hms_example_projects' / 'arf_demo',   ignore_errors=True)
print("Cleaned up")

Cleaned up


## Summary

| Method | Input | Output |
|--------|-------|--------|
| `lookup_arf_from_dar(cda, dar_curve)` | CDA + DAR curve | float ARF |
| `compute_kcda_cdas(basin, junctions)` | Basin file + junction list | DataFrame: junction, CDA, subbasin_count |
| `build_kcda_arf_table(basin, junctions, dar_curve)` | All of the above | DataFrame sorted by CDA with ARF |
| `apply_arf(met_path, arf=scalar)` | Met file + float | Depths modified in-place, `.met.bak` created |

**Key Design Points**:
- `lookup_arf_from_dar` returns **1.0** for CDA at or below the curve minimum
  (no reduction for very small areas)
- `apply_arf(arf=...)` targets the `Precip Method Parameters:` block, *not*
  individual `Subbasin:` blocks, which are empty in Frequency Based Hypothetical
  met files
- The non-destructive clone-then-apply pattern produces one met file per
  analysis junction for side-by-side comparison in the HMS GUI

## Next Steps

- **05_clone_workflow.ipynb** — Non-destructive cloning for QAQC
- **15_upstream_network_analysis.ipynb** — Full upstream network traversal
- **10_atlas14_hyetograph.ipynb** — Generating the point-precipitation depths
  that ARF is applied to